# Example 3.8 / 1: Diagnose an Infeasible Model

Companion notebook for **3.8 Infeasible and Unbounded Results** in the AMPL lab Documentation, section "Diagnose an Infeasible Model". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Diagnose an Infeasible Model

Start with the smallest reproducible instance. Verify input units and signs, then look for
conflicting bounds, totals, and balance equations. In the run below, a demand of 50 tons cannot be
met with a capacity of 40. Changing the data that causes the conflict and solving again is the
simplest diagnostic:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param need >= 0;                    # Tons that must be produced
param cap >= 0;                     # Tons the plant can produce
var Make >= 0;                      # Tons to produce
minimize Cost: Make;                # Produce as little as allowed
subject to Demand: Make >= need;    # At least the demand
subject to Capacity: Make <= cap;   # At most the capacity

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["need"] = 50
ampl.param["cap"] = 40

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
ampl.param["cap"] = 60
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.var["Make"].value())

infeasible
solved 50.0


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
param need := 50;
param cap := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
option solver highs;
option solver_msg 0;
solve;
display solve_result;
let cap := 60;
solve;
display solve_result, Make;
""")
except Exception:
    pass  # AMPL has printed the error above

[WARNING] 
	presolve, variable Make:
		impossible deduced bounds: lower = 50, upper = 40;
		difference = 10


solve_result = infeasible

solve_result = solved
Make = 50

